# Fetch and reduce DataDecide (CPU)

This leg turns the per-instance release into the reduced runs everything else
reads. It is bandwidth-bound rather than compute-bound: 25 recipe tarballs of
about 4.9 GB each, downloaded one at a time, reduced, and deleted before the next
one starts, so peak disk stays near a single tarball.

Session settings: **CPU**, internet **on**, persistence **on** so the reduced runs
survive a restart.

Budget roughly 40 minutes per recipe on a Kaggle CPU session, which is more than a
single session allows, so run it in batches with `--recipes` and let the output
directory accumulate across sessions.

In [ ]:
!git clone -q https://github.com/garyzhang1006/seed-noise.git /kaggle/working/seed-noise
!pip -q install -e /kaggle/working/seed-noise

In [ ]:
import seednoise
from seednoise.data.datadecide import RECIPES, TRAITS

print(seednoise.__version__, len(RECIPES), "recipes,", len(TRAITS), "traits")

## Check the install before spending any bandwidth

The self-test simulates a population whose answer is known and runs every stage on
it. If this does not pass, nothing downstream is worth starting.

In [ ]:
!seednoise selftest --out /kaggle/working/selftest

## One batch of recipes

Edit `BATCH` to the recipes this session should take. The manifest is rewritten
after every recipe, so an interrupted session leaves a readable record of what
finished.

In [ ]:
BATCH = RECIPES[:4]      # one session's worth
print(BATCH)

In [ ]:
!seednoise fetch \
    --recipes {" ".join(BATCH)} \
    --tmp /kaggle/tmp \
    --out /kaggle/working/runs

## What landed

Every reduced run is one `.npz` of float16 margins and bit-packed accuracy, about
80 KB, so the whole population is roughly 30 MB and reloads in a second.

In [ ]:
from pathlib import Path
runs = sorted(Path("/kaggle/working/runs").glob("*.npz"))
print(len(runs), "runs,",
      sum(p.stat().st_size for p in runs) / 1e6, "MB")
print(runs[0].name if runs else "nothing yet")

In [ ]:
from seednoise.store import load_run
items, meta = load_run(runs[0])
print(meta)
print(items.n_items, "items, accuracy", items.correct.mean().round(4))

## Sanity check on one cell

Two runs of the same configuration should agree far more than two runs of
different configurations. If that ordering is reversed, the reduction has gone
wrong and the estimator will happily produce a number anyway.

In [ ]:
import numpy as np
from collections import defaultdict

by_cell = defaultdict(list)
for p in runs:
    it, m = load_run(p)
    by_cell[(m["recipe"], m["size"])].append(it.margin.astype(np.float64))

cells = [v for v in by_cell.values() if len(v) >= 2]
within = np.mean([np.corrcoef(v[0], v[1])[0, 1] for v in cells])
between = np.corrcoef(cells[0][0], cells[-1][0])[0, 1] if len(cells) > 1 else np.nan
print(f"within-cell {within:.4f}  between-cell {between:.4f}")